# Modulo 1: Nivel de aplicacion. Tema 4: DNS

### Domain Name System (DNS)
- Protocolo de nivel de aplicacion para resolver un nombre de dominio
- Utiliza el puerto 53 UDP para comunicacion cliente - servidor
- Utiliza el puerto 53 TCP para comunicacion servidor - servidor

Servicios DNS basicos:
- Traduccion nombre ←→ direccion IP
- Host aliasing: multiples nombres para un mismo host
- Distribucion de carga: Servidores Web replicados: varias direcciones IP corresponden al mismo nombre

NO utiliza un esquema centralizado ya que es un servicio critico para internet (por lo que no puede ser un punto unico de fallo) y porque el volumen de trafico es inmanejable



La ceremonia de firma de la llave de la zona raíz convierte a los servidores de nombres DNS raíz en un anclaje de veracidad. En lugar de que la confianza se derive de una zona primaria, la confianza se asume. Toda esta ceremonia está diseñada para reforzar esa confianza. Es un aspecto muy humano de la seguridad de Internet. La razón por la que puedes confiar en los servidores DNS raíz es porque puedes confiar en las personas que los firman. Y la razón por la que puedes confiar en ellas es por los estrictos protocolos que siguen al hacerlo. En eso consiste la ceremonia de firma de la llave de la zona raíz.

<img src="ServidoresDNS.png">

### Servidores raiz

Ultimo recurso para servidores DNS, que no conocen a otros servidores que necesitan.
Tienen una funcion esencial para Internet. Basicamente no funcionaria sin ellos
Son gestionados por la ICANN (Internet Corporation for Assigned Names and Numbers)

#### DNS digging

dig es una herramienta que permite hacer todo tipo de consultas dns

**1. Resolucion simple:**

In [ ]:
!dig uam.es


; <<>> DiG 9.18.30-0ubuntu0.24.04.2-Ubuntu <<>> uam.es
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 47690
;; flags: qr rd ra; QUERY: 1, ANSWER: 1, AUTHORITY: 0, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 65494
;; QUESTION SECTION:
;uam.es.				IN	A

;; ANSWER SECTION:
uam.es.			10800	IN	A	150.244.214.237

;; Query time: 4 msec
;; SERVER: 127.0.0.53#53(127.0.0.53) (UDP)
;; WHEN: Sun Apr 13 16:51:50 CEST 2025
;; MSG SIZE  rcvd: 51



**2. Resolucion completa, hasta los servidores raiz:**

In [4]:
!dig +trace uam.es


; <<>> DiG 9.18.30-0ubuntu0.24.04.2-Ubuntu <<>> +trace uam.es
;; global options: +cmd
.			420	IN	NS	f.root-servers.net.
.			420	IN	NS	j.root-servers.net.
.			420	IN	NS	a.root-servers.net.
.			420	IN	NS	l.root-servers.net.
.			420	IN	NS	g.root-servers.net.
.			420	IN	NS	c.root-servers.net.
.			420	IN	NS	d.root-servers.net.
.			420	IN	NS	b.root-servers.net.
.			420	IN	NS	e.root-servers.net.
.			420	IN	NS	h.root-servers.net.
.			420	IN	NS	m.root-servers.net.
.			420	IN	NS	i.root-servers.net.
.			420	IN	NS	k.root-servers.net.
;; Received 811 bytes from 127.0.0.53#53(127.0.0.53) in 4 ms

es.			172800	IN	NS	g.nic.es.
es.			172800	IN	NS	a.nic.es.
es.			172800	IN	NS	c.nic.es.
es.			172800	IN	NS	h.nic.es.
es.			86400	IN	DS	54404 8 2 C130DF280CD8EBB99C8B6C8335B9FD5F1D709641494E044C8085B1AC 21833B47
es.			86400	IN	RRSIG	DS 8 1 86400 20250426050000 20250413040000 53148 . Nv5rHciDwxKt0fPokBgWMb1fJNYqXR320z5/YSjdEdh2O5SmFEVQFiCZ 1i1cT43bfaUX6E4XBYDRjynIrsN24FQ6SfJZefDmoyln9MEnAjCsxWss rf6e2iVc4hjHi

Lo que se puede ver en esta ejecucion es una serie de pasos para la resolucion del nombre uam.es:
1. Consulta a los servidores raiz(.) quien sabe sobre el dominio .es. Estos le responden con los servidores autoritativos para .es (por ejemplo a.nic.es)
2. Consulta a los servidores del TLD .es (es.). Ahora pregunta a esos servidores (*.nic.es) quienes tienen la informacion para uam.es
3. Consulta final a los servidores de uam.es (uam.es.). Ya con los servidores autoritativos para uam.es, se les pregunta directamente cual es la IP, a lo que responden con la IP 150.244.214.237

**Formato general de cada linea con resultado DNS:**

[Nombre]   [TTL]   [Clase]   [Tipo]   [Datos]

**Tipos de registros que aparecen en el output:**
| Tipo | Significado |
| -----|-------------|
|**NS Name Server** | indica qué servidor DNS es autoritativo para el dominio.|
|**A Address**|devuelve la dirección IPv4 del dominio.|
|**CNAME Canonical NAME**|Sirven para crear "alias" (punteros) diferentes para un mismo dominio|
|**MX eMailer eXchange**|Permiten obtener el servidor de correo asociado a un dominio|
|DS Delegation Signer|parte de DNSSEC, contiene la clave que enlaza zonas firmadas.|
|RRSIG Resource Record Signature|firma criptográfica de un conjunto de registros, también parte de DNSSEC.|
|NSEC3|Proporciona prueba de que un nombre de dominio no existe, usado en DNSSEC para seguridad.|

Si te fijas, al consultar los servidores raiz aparece que se reciben 811 bytes de la direccion 127.0.0.53. Esta direccion corresponde a una direccion IP local usada por el resolver DNS local de sistemas Linux modernos. Esta direccion es manejada por un servicio llamado systemd-resolved. 
A no ser que se especifique el servidor (con @), primero se conusltara al resolvedor local (en este caso 128.0.0.53). Este resolvedor puede tener:
- cache
- configuraciones especiales
- resoluciones parciales

Pero al usar +trace, dig ignora el comportamiento normal del sistem ay nace la resolucion el mismo paso por paso, pero la primera llamada aun pasa por el resolvedor local para obtener la lista de root servers que ya estan preconfigurados ahi.

Puedes ver su configuración así:

In [6]:
!resolvectl status

Global
         Protocols: -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported
  resolv.conf mode: stub

Link 2 (eno1)
    Current Scopes: DNS
         Protocols: +DefaultRoute -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported
Current DNS Server: 100.100.1.1
       DNS Servers: 100.100.1.1 100.90.1.1

Link 3 (wlo1)
    Current Scopes: none
         Protocols: -DefaultRoute -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported

Link 4 (docker0)
    Current Scopes: none
         Protocols: -DefaultRoute -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported


Como ahora mismo estoy conectado por cable se puede ver que las demas interfaces estan deshabilitadas. Podemos ver algo de informacion sobre la configuracion y servidores DNS:
- LLMNR / -mDNS: significa que no estás usando resolución local por multicast (como la que se usa en redes locales tipo .local).
- DNSSEC=no/unsupported: tu sistema no valida DNSSEC (ni lo tiene habilitado ni soportado).
- resolv.conf mode: stub: el archivo /etc/resolv.conf apunta al resolvedor local (127.0.0.53), que actúa como "puente".

En Link 2 (eno1) se puede ver mi interfaz que se esta usando actualmente. Mis servidores DNS reales (recibidos por DHCP probablemente) son:
- 100.100.1.1
- 100.90.1.1
Las consultas primero van al resolvedor local (127.0.0.53), y luego este las reenvia a estos servidores

**3. Se puede forzar a utilizar un servidor DNS concreto:**

In [ ]:
# DNS Cloudflare: 1.1.1.1
!dig @8.8.8.8 +trace uam.es


; <<>> DiG 9.18.30-0ubuntu0.24.04.2-Ubuntu <<>> @8.8.8.8 +trace uam.es
; (1 server found)
;; global options: +cmd
.			87203	IN	NS	f.root-servers.net.
.			87203	IN	NS	b.root-servers.net.
.			87203	IN	NS	l.root-servers.net.
.			87203	IN	NS	j.root-servers.net.
.			87203	IN	NS	e.root-servers.net.
.			87203	IN	NS	d.root-servers.net.
.			87203	IN	NS	m.root-servers.net.
.			87203	IN	NS	k.root-servers.net.
.			87203	IN	NS	c.root-servers.net.
.			87203	IN	NS	i.root-servers.net.
.			87203	IN	NS	a.root-servers.net.
.			87203	IN	NS	h.root-servers.net.
.			87203	IN	NS	g.root-servers.net.
.			87203	IN	RRSIG	NS 8 0 518400 20250426050000 20250413040000 53148 . eygqva3gcE/OWez/gnKErbvibPwO8W7nYrSKQl2ZpWowdvjtVQJMMPTO KoXdfDGHKCXNSvu3mu/sW0x5oCtUSvs0CeX0bGIRqXQsuzG8JYww1qSq dP8MFXpCe6wKLaanaAZbc+OZa7VRsOSL0rHJo1/voIJvlhg2AzkbAjb3 yjfDVG3/LrrOk7PbZ1kWG/9yuVdREwva6AfejL+XxNWYHDxP4/67KnKU Xb3cxBHvjCQm4MHkUqVVC2dwhIRLKa8x/SpZkW/pCzzdzR3elNfgnvXy LGk2eBC81oSsBQOYGsGWONUeX/Et+sbo2q5J5I9qirPBxMGkgo1u5pH/ 0ZJ5

**4. Encontrar el servidor de correo de un dominio:**

In [7]:
!dig MX uam.es


; <<>> DiG 9.18.30-0ubuntu0.24.04.2-Ubuntu <<>> MX uam.es
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 62742
;; flags: qr rd ra; QUERY: 1, ANSWER: 2, AUTHORITY: 0, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 65494
;; QUESTION SECTION:
;uam.es.				IN	MX

;; ANSWER SECTION:
uam.es.			300	IN	MX	10 mxb-006a4e02.gslb.pphosted.com.
uam.es.			300	IN	MX	10 mxa-006a4e02.gslb.pphosted.com.

;; Query time: 4 msec
;; SERVER: 127.0.0.53#53(127.0.0.53) (UDP)
;; WHEN: Sun Apr 13 17:25:50 CEST 2025
;; MSG SIZE  rcvd: 110



Este resultado dice exactamente como maneja el dominio uam.es su correo electronico usando registros MX (Mail Exchange)
Un registro MX (Mail Exchange) indica a qué servidores debe enviarse el correo electrónico dirigido a un dominio específico. Por ejemplo, si alguien escribe a usuario@uam.es, el servidor que envía el correo necesita saber a qué dirección enviar ese correo, y eso se determina consultando el registro MX del dominio.

La salida: 
;; ANSWER SECTION:
uam.es.			300	IN	MX	10 mxb-006a4e02.gslb.pphosted.com.
uam.es.			300	IN	MX	10 mxa-006a4e02.gslb.pphosted.com.
quiere decir que:
- El correo de uam.es es manejado por dos servidores:
    - mxb-006a4e02.gslb.pphosted.com.
    - mxa-006a4e02.gslb.pphosted.com.
- Ambos tienen prioridad 10 (el numero que va antes del nombre)
    - Si uno falla, el otro puede actuar como respaldo, o pueden hacer balanceo de carga
- Estos dominios están bajo el subdominio pphosted.com, lo que indica que la UAM externaliza su correo electrónico a Proofpoint (una empresa especializada en servicios de correo y filtrado de seguridad).

In [8]:
# En esta query, el servidore de correo no existe
!dig MX ii.uam.es


; <<>> DiG 9.18.30-0ubuntu0.24.04.2-Ubuntu <<>> MX ii.uam.es
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 57230
;; flags: qr rd ra; QUERY: 1, ANSWER: 0, AUTHORITY: 1, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 65494
;; QUESTION SECTION:
;ii.uam.es.			IN	MX

;; AUTHORITY SECTION:
uam.es.			10800	IN	SOA	dns4.uam.es. hostmaster.uam.es. 2014076393 21600 7200 2592000 10800

;; Query time: 10 msec
;; SERVER: 127.0.0.53#53(127.0.0.53) (UDP)
;; WHEN: Sun Apr 13 17:26:26 CEST 2025
;; MSG SIZE  rcvd: 90



<img src="Dominios.png">

### Proceso de resolucion
0. Navegador solicita al resolver (proceso local) la resolución.
1. El resolver se comunica con el DNS local (normalmente, configurado a mano). Éste comprueba si la petición está en su caché. Como es una petición recursiva, él se encarga del resto del proceso. 
2. Si no conoce la IP del TLD correspondiente (.es), pregunta a un servidor raiz aleatorio. Si la conoce, pregunta al TLD directamente (paso 4)
3. El servidor raiz devuelve una lista con TLDs para el dominio .es
4. El servidor DNS elige una entrada de la lista y la consulta
5. El servidor TLD correspondiente responde con la direccion IP del servidor primario del dominio .rediris.es
6. El servidor DNS pregunta finalmente al servidor primario por el host 'www'
7. Se devuelve la respuesta correcta, que se pasa al solicitante en el paso 8

<img src="Procesoderesolucion.png" style="display: block; margin: auto;">

Resolver: se ha mencionado antes pero es un proceso local que se encarga de recibir las peticiones de resolucion DNS de todas las aplicaciones. Cada ISP tiene su servidor DNS local.

In [4]:
# Para encontrar el servidor DNS local:
# MacOS: % scutil --dns
# Windows: >ipconfig /all
# Linux: nmcli device show <<interface>>
!nmcli device show eno1 | grep -v -E "IP6\.ADDRESS\[1\]|IP6\.ADDRESS\[2\]|IP6\.ADDRESS\[3\]|IP6\.GATEWAY|IP6\.ROUTE\[1\]|IP6\.ROUTE\[2\]|IP6\.ROUTE\[3\]|GENERAL.HWADDR"

GENERAL.DEVICE:                         eno1
GENERAL.TYPE:                           ethernet
GENERAL.MTU:                            1500
GENERAL.STATE:                          100 (connected)
GENERAL.CONNECTION:                     netplan-eno1
GENERAL.CON-PATH:                       /org/freedesktop/NetworkManager/ActiveConnection/3
WIRED-PROPERTIES.CARRIER:               on
IP4.ADDRESS[1]:                         192.168.1.138/24
IP4.GATEWAY:                            192.168.1.1
IP4.ROUTE[1]:                           dst = 192.168.1.0/24, nh = 0.0.0.0, mt = 100
IP4.ROUTE[2]:                           dst = 0.0.0.0/0, nh = 192.168.1.1, mt = 100
IP4.DNS[1]:                             100.90.1.1
IP4.DNS[2]:                             100.100.1.1


### Caché DNS

La infraestructura DNS cachea a varios niveles las traducciones para reducir los tiempos de respuesta:

<img src="CacheDNS.png">

Las entradas cacheadas DEBEN caducat (tienen que tener un TTL Time To Live):
- Si no, si una maquina cambia su IP, los DNS proporcionaran siempre respuestas incorrectas
- Un TTL tipico para servidores DNS es de 48 horas.

### Formato de los mensajes

Tanto consultas como respuestas DNS tienen el mismo formato

**Header (Cabecera)**
|Campo|Tamaño|Descripcion|
|-----|------|-----------|
|ID|16 bits|identificador unico para que el cliente pueda mapear la respuesta|
|Flags|16 bits|Si es consulta o respuesta, recursividad o respuesta autoritaria|
|QDCOUNT|16 bits|Número de preguntas (normalmente 1)|
|ANCOUNT|16 bits|Número de respuestas (0 en consulta)|
|NSCOUNT|16 bits|Numero de registros de autoridad|
|ARCOUNT|16 bits|Numero de registros adicionales|

**Question (Pregunta)**
|Campo|Descripcion|
|-----|-----------|
|QNAME|Nombre del dominio (ej. uam.es)|
|QTYPE|Tipo de registro (A, MX, NS, etc.)|
|QCLASS|Clase (normalmente IN para Internet)|

**Answer (Respuesta)**
|Campo|Descripcion|
|-----|-----------|
|NAME|Nombre del dominio|
|TYPE|Tipo de registro (A, MX, etc.)|
|CLASS|Clase (IN)|
|TTL|Time To Live (segundos)|
|RDLENGTH|Longitud de los datos|
|RDATA|Datos (por ejemplo, IP o nombre del servidor de correo)|

**Authority (Autoridad)**
Indica que servidores tienen autoridad sobre el dominio (util si el servidor no tiene la respuesta directamente).

**Additional (Adicional)**
Contiene datos extra utiles, como registros A para los servidores de autoridad (para evitar consultas adicionales).

<img src="Formatodelosmensajes.png" style="display: block; margin: auto;">

### Generacion de nuevos dominios

Ejemplo: La empresa **"UAM Utopia"** quiere crear su sitio web con el dominio: ```uamutopia.com```

**1. Registrar el nombre de dominio**  
Esto se hace a traves de un registrado DNS (como GoDaddy, Namecheap, etc.)  
¿Que hace el registrador?
- Recoge los datos personales del titular del domino
- Registra el dominio ```uamutopia.com``` y crea algunos registros DNS basicos como:
    - `uamutopia.com` → `212.212.212.1` → tipo A  
  (esto dice: "la IP del sitio web es 212.212.212.1")
    - `dns1.uamutopia.com` → `212.212.212.1` → tipo NS  
    (esto dice: "este servior es el que gestiona los nombres para el dominio")  

**TLD** significa "Top Level Domain" en este caso `.com`

**2. Instalar y configurar un servidor DNS propio**  
Ahora que el dominio esta registrado, necesitas configurar un servidor DNS real que responda consultas sobre tu dominio  
¿Que hay que hacer?
- Instalar un servidor DNS (como BIND, PowerDNS…) en una máquina que tenga la IP 212.212.212.1.
- Configurar los registros dentro del servidor DNS para que funcionen correctamente
- Ejemplo de registros a configurar:
    - Registro A:
        - `www.uamutopia.com` → `212.212.212.1` → Esto permite acceder a la web
    - Registro MX:
        - `uamutopia.com` → (servidor de correo) → Esto permite recibir y enviar correos electronicos con direcciones como `info@uamutopia.com`

### Seguridad
**Ataques DDoS (Distributed Denial of Service)**
- Inundar servidores raiz con trafico
    - Sin exito hasta la fecha
    - Contramedida: servidores locales cachean las IPs de los TLDs, evitando consultas a los servidores raiz

**Ataques de suplantacion (spoofing)**
- Interceptan las peticiones DNS (que no estan protegidas criptograficamente), y devuelven respuestas incorrectas, para redirigir el trafico a un servidor del atacante
    - Contramedida: RFC 4033 (Servicios de autenticacion DNSSEC)

#### DNS spoofing
- El atacante debe:
    - Ser mas rapido en su respuesta que el servidor real
    - Adivinar el ID de la transaccion original
    - La peticion no debe estar ya cacheada por el DNS local

<img src="DNSspoofing.png" width="400" style="display: block; margin: auto;">

<style>
    table { page-break-inside: avoid; }
    pre { page-break-inside: avoid; }
</style>